# augmentations

This reads processed training samples from `metadata.csv`, applies augmentation to image-mask pairs, and saves preview outputs.

## Imports and project paths

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import rasterio
import albumentations as A

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p

    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p

    return start

PROJECT_ROOT = find_project_root()
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "augmentations"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("METADATA_PATH:", METADATA_PATH)
print("OUT_DIR:", OUT_DIR)

PROJECT_ROOT: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project
METADATA_PATH: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\data\processed\metadata.csv
OUT_DIR: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\outputs\augmentations


## Path helpers and image loading

In [ ]:
def normalize_rel_path(p):
    p = Path(str(p).replace("\\", "/"))
    parts = list(p.parts)
    if parts and parts[0] == "src":
        p = Path(*parts[1:])
    return p

def to_abs_path(p):
    p = normalize_rel_path(p)
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / p).resolve()

def to_rel_str(p):
    p = normalize_rel_path(p)
    if p.is_absolute():
        try:
            return str(p.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/")
        except Exception:
            return str(p).replace("\\", "/")
    return str(p).replace("\\", "/")

def percentile_stretch(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    if arr.ndim == 2:
        lo, hi = np.percentile(arr, (2, 98))
        return np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)

    out = np.zeros_like(arr, dtype=np.float32)
    for c in range(arr.shape[2]):
        lo, hi = np.percentile(arr[:, :, c], (2, 98))
        out[:, :, c] = np.clip((arr[:, :, c] - lo) / (hi - lo + 1e-6), 0, 1)
    return out

def read_tif_rgb(path_str: str) -> np.ndarray:
    path = to_abs_path(path_str)
    with rasterio.open(path) as src:
        if src.count >= 3:
            arr = src.read([1, 2, 3])
            arr = np.transpose(arr, (1, 2, 0))
        else:
            arr = src.read(1)
            arr = np.stack([arr] * 3, axis=-1)

    arr = percentile_stretch(arr)
    return (arr * 255).astype(np.uint8)

def read_mask_png(path_str: str) -> np.ndarray:
    path = to_abs_path(path_str)
    mask = Image.open(path).convert("L")
    mask = np.array(mask, dtype=np.uint8)
    return (mask > 0).astype(np.uint8)


## Augmentation and saving helpers

In [ ]:
def build_train_transform():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
    ])

def save_img(img: np.ndarray, out_path: Path, cmap=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(6, 6))
    if img.ndim == 2:
        plt.imshow(img, cmap=cmap or "gray")
    else:
        plt.imshow(img)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight", pad_inches=0)
    plt.close()

def save_overlay(image: np.ndarray, mask: np.ndarray, out_path: Path, color=(0, 1, 1), alpha=0.35):
    img = image.astype(np.float32) / 255.0
    color = np.array(color, dtype=np.float32).reshape(1, 1, 3)
    overlay = np.where(mask[..., None] > 0, (1 - alpha) * img + alpha * color, img)
    overlay = np.clip(overlay, 0, 1)
    save_img((overlay * 255).astype(np.uint8), out_path)


## Generate augmentation previews

In [ ]:
df = pd.read_csv(METADATA_PATH)

required_cols = {"sample_id", "split", "post_path", "mask_path"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"metadata.csv is missing required columns: {missing}")

if "mask_status" in df.columns:
    df = df[df["mask_status"] == "OK"].copy()

train_df = df[df["split"] == "train"].copy()
if train_df.empty:
    raise ValueError("No training rows found in metadata.csv")

n_samples = min(10, len(train_df))
sample_df = train_df.sample(n=n_samples, random_state=42)

transform = build_train_transform()
preview_rows = []

print("Generating augmentation previews for", n_samples, "training samples")

for row in sample_df.itertuples(index=False):
    sample_id = row.sample_id

    image = read_tif_rgb(row.post_path)
    mask = read_mask_png(row.mask_path)

    augmented = transform(image=image, mask=mask)
    aug_image = augmented["image"]
    aug_mask = augmented["mask"]

    orig_img_out = OUT_DIR / f"{sample_id}_orig_post.png"
    orig_mask_out = OUT_DIR / f"{sample_id}_orig_mask.png"
    aug_img_out = OUT_DIR / f"{sample_id}_aug_post.png"
    aug_mask_out = OUT_DIR / f"{sample_id}_aug_mask.png"
    orig_overlay_out = OUT_DIR / f"{sample_id}_orig_overlay.png"
    aug_overlay_out = OUT_DIR / f"{sample_id}_aug_overlay.png"

    save_img(image, orig_img_out)
    save_img(mask * 255, orig_mask_out, cmap="gray")
    save_overlay(image, mask, orig_overlay_out)

    save_img(aug_image, aug_img_out)
    save_img(aug_mask * 255, aug_mask_out, cmap="gray")
    save_overlay(aug_image, aug_mask, aug_overlay_out)

    preview_rows.append({
        "sample_id": sample_id,
        "orig_post_preview": to_rel_str(orig_img_out),
        "orig_mask_preview": to_rel_str(orig_mask_out),
        "aug_post_preview": to_rel_str(aug_img_out),
        "aug_mask_preview": to_rel_str(aug_mask_out),
        "orig_overlay_preview": to_rel_str(orig_overlay_out),
        "aug_overlay_preview": to_rel_str(aug_overlay_out),
    })

preview_df = pd.DataFrame(preview_rows)
preview_csv = OUT_DIR / "augmentation_preview_summary.csv"
preview_df.to_csv(preview_csv, index=False)

print("Saved preview summary:", preview_csv)
display(preview_df.head())


Generating augmentation previews for 10 training samples
Saved preview summary: C:\Users\Rajesh\Desktop\MSc-II\AI_FOR_SPACE\flood_project\outputs\augmentations\augmentation_preview_summary.csv


,sample_id,orig_post_preview,orig_mask_preview,aug_post_preview,aug_mask_preview,orig_overlay_preview,aug_overlay_preview
0,sample_00081,outputs/augmentations/sample_00081_orig_post.png,outputs/augmentations/sample_00081_orig_mask.png,outputs/augmentations/sample_00081_aug_post.png,outputs/augmentations/sample_00081_aug_mask.png,outputs/augmentations/sample_00081_orig_overla...,outputs/augmentations/sample_00081_aug_overlay...
1,sample_00198,outputs/augmentations/sample_00198_orig_post.png,outputs/augmentations/sample_00198_orig_mask.png,outputs/augmentations/sample_00198_aug_post.png,outputs/augmentations/sample_00198_aug_mask.png,outputs/augmentations/sample_00198_orig_overla...,outputs/augmentations/sample_00198_aug_overlay...
2,sample_00101,outputs/augmentations/sample_00101_orig_post.png,outputs/augmentations/sample_00101_orig_mask.png,outputs/augmentations/sample_00101_aug_post.png,outputs/augmentations/sample_00101_aug_mask.png,outputs/augmentations/sample_00101_orig_overla...,outputs/augmentations/sample_00101_aug_overlay...
3,sample_00181,outputs/augmentations/sample_00181_orig_post.png,outputs/augmentations/sample_00181_orig_mask.png,outputs/augmentations/sample_00181_aug_post.png,outputs/augmentations/sample_00181_aug_mask.png,outputs/augmentations/sample_00181_orig_overla...,outputs/augmentations/sample_00181_aug_overlay...
4,sample_00059,outputs/augmentations/sample_00059_orig_post.png,outputs/augmentations/sample_00059_orig_mask.png,outputs/augmentations/sample_00059_aug_post.png,outputs/augmentations/sample_00059_aug_mask.png,outputs/augmentations/sample_00059_orig_overla...,outputs/augmentations/sample_00059_aug_overlay...
